# Step 3: SageMaker Fine-tuning 실행

SageMaker Training Job을 사용하여 한국인 딥페이크 데이터(KoDF)로 모델을 Fine-tuning합니다.

## 실습 목표
- SageMaker PyTorch Estimator 구성
- **SageMaker Experiments로 실험 추적**
- Training Job 실행 (Spot Instance 옵션)
- 학습 모니터링

## 3.1 환경 설정

In [ ]:
import json
import os
import sagemaker
from sagemaker.pytorch import PyTorch
from sagemaker.experiments.run import Run
from datetime import datetime
from pathlib import Path

# ============================================
# 프로젝트 경로 자동 설정
# ============================================
home_dir = Path.home()
PROJECT_ROOT = home_dir / 'deepfake-detection-sagemaker'
notebook_dir = PROJECT_ROOT / '3_fine_tuning'
os.chdir(notebook_dir)

print(f"Project Root: {PROJECT_ROOT}")
print(f"Current Dir: {os.getcwd()}")

# 설정 로드
config_path = PROJECT_ROOT / 'config.json'
with open(config_path, 'r') as f:
    config = json.load(f)

# SageMaker 세션
sagemaker_session = sagemaker.Session()
role = config['role']
bucket = config['bucket']
prefix = config['prefix']

# Experiment 설정
EXPERIMENT_NAME = "deepfake-detection-kodf"
RUN_NAME = f"finetuning-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

print(f"Role: {role[:50]}...")
print(f"Bucket: {bucket}")
print(f"Experiment: {EXPERIMENT_NAME}")

## 3.2 하이퍼파라미터 설정

In [ ]:
# 100-200 레벨에 맞춘 간단한 하이퍼파라미터
hyperparameters = {
    'epochs': 5,              # 빠른 실습을 위해 5 에포크
    'batch-size': 32,
    'learning-rate': 0.0001,
    'model-name': 'efficientnet_b0'
}

print("하이퍼파라미터:")
for k, v in hyperparameters.items():
    print(f"  {k}: {v}")

## 3.3 PyTorch Estimator 생성

In [ ]:
# Spot Instance 사용 여부 (비용 ~70% 절감)
USE_SPOT = True

# SageMaker PyTorch Estimator
estimator = PyTorch(
    entry_point='train.py',
    source_dir='.',
    role=role,
    instance_count=1,
    instance_type='ml.g4dn.xlarge',
    framework_version='2.0.0',
    py_version='py310',
    hyperparameters=hyperparameters,
    output_path=f's3://{bucket}/{prefix}/output',
    sagemaker_session=sagemaker_session,
    # Spot Instance 설정 (비용 절감)
    use_spot_instances=USE_SPOT,
    max_wait=7200 if USE_SPOT else None,
    max_run=3600,
)

print("PyTorch Estimator 생성 완료")
print(f"Instance Type: ml.g4dn.xlarge")
print(f"Spot Instance: {'활성화 (비용 ~70% 절감)' if USE_SPOT else '비활성화'}")
print(f"Framework: PyTorch 2.0.0")

## 3.4 Training Job 실행

In [ ]:
# 데이터 채널 설정
data_channels = {
    'train': config['s3_train_path'],
    'val': config['s3_val_path']
}

print("데이터 채널:")
for k, v in data_channels.items():
    print(f"  {k}: {v}")

In [ ]:
# SageMaker Experiments와 함께 Training Job 시작
print("=" * 60)
print("  SageMaker Training Job 시작")
print("  한국인 딥페이크 데이터로 Fine-tuning 중...")
print(f"  Experiment: {EXPERIMENT_NAME}")
print("=" * 60)

with Run(
    experiment_name=EXPERIMENT_NAME,
    run_name=RUN_NAME,
    sagemaker_session=sagemaker_session
) as run:
    # 하이퍼파라미터 로깅
    run.log_parameters(hyperparameters)
    run.log_parameter("instance_type", "ml.g4dn.xlarge")
    run.log_parameter("use_spot", USE_SPOT)
    
    # Training 실행
    estimator.fit(data_channels, wait=True, logs='All')
    
    # 학습 완료 후 메트릭 로깅 (나중에 평가 결과로 업데이트)
    run.log_parameter("model_data", estimator.model_data)

## 3.5 학습 결과 확인

In [ ]:
# 학습된 모델 경로
model_data = estimator.model_data
print(f"학습된 모델 위치: {model_data}")

# Training Job 이름 저장
training_job_name = estimator.latest_training_job.name
print(f"Training Job 이름: {training_job_name}")

# 설정 업데이트
config['model_data'] = model_data
config['training_job_name'] = training_job_name

config_path = PROJECT_ROOT / 'config.json'
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

print(f"설정이 업데이트되었습니다: {config_path}")

In [ ]:
# 학습 메트릭 확인 (CloudWatch)
from sagemaker.analytics import TrainingJobAnalytics

try:
    training_analytics = TrainingJobAnalytics(training_job_name)
    df = training_analytics.dataframe()
    print("학습 메트릭:")
    display(df)
except Exception as e:
    print(f"메트릭 조회 실패 (정상일 수 있음): {e}")

## 완료!

SageMaker Fine-tuning이 완료되었습니다.

**결과:**
- 한국인 딥페이크 데이터(KoDF)로 모델이 Fine-tuning됨
- 학습된 모델이 S3에 저장됨

**➡️ 다음 단계: `4_after_evaluation/evaluate_after.ipynb`**

Fine-tuned 모델의 성능을 평가해봅니다!